# NB09: Preprocessing Pipeline Overview

> **Local pipeline required.** This notebook inspects derivatives generated by the processing pipeline (FreeSurfer, fMRIPrep, QSIPrep, MRtrix3) and cannot run in Google Colab. See NB00--NB06 and NB08 for Colab-compatible notebooks.

## Learning Objectives
- Understand the full multimodal MRI preprocessing pipeline for glioma digital twins
- Learn why each step (FreeSurfer, fMRIPrep, QSIPrep, MRtrix3) is needed
- See how all modalities are co-registered into a common T1w space
- Understand the connection between preprocessing outputs and brain mechanics modeling
- Learn how longitudinal (SyN) registration produces deformation fields for FE validation
- Know how to run the pipeline and inspect outputs

## Overview

Building a patient-specific brain digital twin requires combining structural,
diffusion, and functional MRI into a unified coordinate frame. This notebook
provides a didactic walkthrough of the 6-step preprocessing pipeline:

| Step | Tool | Input | Output | Brain Mechanics Role |
|------|------|-------|--------|---------------------|
| 1 | FreeSurfer 8.1.0 | T1w | Surfaces, parcellation | FE mesh geometry |
| 2 | fMRIPrep 25.2.4 | BOLD | Preprocessed fMRI | Functional connectivity |
| 3 | QSIPrep 1.1.1 | DWI (AP+PA) | Preprocessed DWI | Input for CSD/DTI |
| 4 | MRtrix3 3.0.8 | Preprocessed DWI | fODFs, DTI maps, tractogram | Material properties |
| 5 | FSL + ANTs | All derivatives | T1w-space outputs | Unified coordinate frame |
| 5b | ANTs SyN | Pre-op + post-op T1w | Warp fields, Jacobian, displacement | Empirical deformation data |
| 6 | Python | Pre/post T1w outputs | Longitudinal comparison | Tissue change mapping |

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

if 'google.colab' in sys.modules:
    REPO_ROOT = Path('/content/glioma-brain-digital-twin')
else:
    _cwd = Path.cwd().resolve()
    REPO_ROOT = _cwd if (_cwd / 'pyproject.toml').exists() else _cwd.parent

BIDS_ROOT = REPO_ROOT / 'data' / 'BIDS_preop_postop'
DERIV_ROOT = REPO_ROOT / 'derivatives'
sys.path.insert(0, str(REPO_ROOT))

SUBJECTS = ['PAT16']
SESSIONS = ['preop', 'postop']

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f'Repository root: {REPO_ROOT}')
print(f'BIDS root:       {BIDS_ROOT}')
print(f'Derivatives:     {DERIV_ROOT}')

Repository root: <REPO_ROOT>
BIDS root:       <REPO_ROOT>/data/BIDS_preop_postop
Derivatives:     <REPO_ROOT>/derivatives


## 1. The Coordinate Space Problem

Each preprocessing tool produces outputs in its own coordinate space:

| Tool | Space | Grid | Orientation |
|------|-------|------|-------------|
| FreeSurfer | Conformed | 256 x 256 x 256 (1 mm) | RAS-permuted |
| fMRIPrep | Native T1w | Scanner-dependent | Scanner orientation |
| QSIPrep | ACPC | ~160 x 200 x 170 (1 mm) | Axis-aligned |

For brain mechanics, we need everything in **one** coordinate frame: the
fMRIPrep native T1w space. This is why Step 5 (co-registration) is critical.

The rigid 6-DOF transform from ACPC to T1w space is computed using FLIRT,
then applied to DTI maps, fODs (with spherical harmonic reorientation),
and tractography streamlines.

## 2. Step 1: FreeSurfer Segmentation

FreeSurfer's `recon-all` extracts cortical surfaces and parcellates the
brain into anatomical regions (Desikan-Killiany atlas). For brain mechanics:

- **Cortical surfaces** (`lh.white`, `lh.pial`) define the geometry for FE mesh generation
- **aparc+aseg** labels identify tissue types (GM, WM, CSF, subcortical structures)
- **Additional segmentations** (brainstem, thalamic nuclei, hippocampal subfields) provide finer mechanical compartments

### Standard vs. Clinical recon-all

For glioma patients, standard `recon-all` may fail near tumors and
resection cavities because it relies on atlas-based priors that assume
normal anatomy. `recon-all-clinical` (FreeSurfer >= 7.4) uses:

- **SynthSeg**: deep learning segmentation robust to pathology
- **SynthDist**: distance-based surface placement (not intensity-driven)
- **SynthSR**: synthesized 1mm MPRAGE for visualization

| Scenario | Recommended tool |
|----------|------------------|
| Post-op (resection cavity) | `recon-all-clinical` (always) |
| Pre-op with large tumor | `recon-all-clinical` (recommended) |
| Pre-op with small/diffuse tumor | Either; standard gives better cortical thickness |

### Running FreeSurfer

```bash
# Standard recon-all (normal anatomy, better cortical thickness)
bash scripts/run_freesurfer.sh --subject PAT16 --session preop --threads 8

# Clinical recon-all (pathological anatomy -- tumors, resection cavities)
bash scripts/run_freesurfer_clinical.sh --subject PAT16 --session postop --threads 8

# Full pipeline with clinical mode for all subjects
bash scripts/run_pipeline.sh --clinical --threads 8
```

### Inspecting with Freeview

```bash
bash scripts/freeview_freesurfer.sh --subject PAT16 --session preop
```

In [2]:
# Check FreeSurfer outputs for each subject x session
fs_dir = DERIV_ROOT / 'freesurfer'
print('FreeSurfer outputs:')
print('=' * 60)
for sub in SUBJECTS:
    for ses in SESSIONS:
        fs_subj = fs_dir / f'sub-{sub}_ses-{ses}'
        aparc = fs_subj / 'mri' / 'aparc+aseg.mgz'
        lh_pial = fs_subj / 'surf' / 'lh.pial'
        status = 'DONE' if aparc.exists() else 'not yet run'
        print(f'  sub-{sub} ses-{ses}: {status}')
        if aparc.exists():
            print(f'    aparc+aseg : {aparc}')
            print(f'    lh.pial   : {"exists" if lh_pial.exists() else "missing"}')

FreeSurfer outputs:
  sub-PAT16 ses-preop: DONE
    aparc+aseg : <REPO_ROOT>/derivatives/freesurfer/sub-PAT16_ses-preop/mri/aparc+aseg.mgz
    lh.pial   : exists
  sub-PAT16 ses-postop: DONE
    aparc+aseg : <REPO_ROOT>/derivatives/freesurfer/sub-PAT16_ses-postop/mri/aparc+aseg.mgz
    lh.pial   : exists


## 3. Step 2: fMRIPrep BOLD Preprocessing

fMRIPrep applies a standardized pipeline to the resting-state fMRI data:

1. **Skull stripping** and bias field correction of the T1w
2. **BOLD-to-T1w coregistration** using boundary-based registration (BBR)
3. **Distortion correction** using SyN (fieldmap-less, since BOLD has single PE direction)
4. **Confound estimation**: head motion, aCompCor, framewise displacement
5. **Resampling** to T1w space and MNI space

For brain mechanics, the T1w-space BOLD output provides:
- Functional connectivity matrices for network analysis
- ROI time series for hemodynamic response estimation
- Functional activation maps co-registered with the FE mesh

### Running fMRIPrep

```bash
bash scripts/run_fmriprep.sh --subject PAT16 --session preop --nthreads 8
```

In [3]:
# Check fMRIPrep outputs
fmriprep_dir = DERIV_ROOT / 'fmriprep'
print('fMRIPrep outputs:')
print('=' * 60)
for sub in SUBJECTS:
    for ses in SESSIONS:
        func_dir = fmriprep_dir / f'sub-{sub}' / f'ses-{ses}' / 'func'
        bold_t1w = list(func_dir.glob('*_space-T1w_desc-preproc_bold.nii.gz')) if func_dir.exists() else []
        confounds = list(func_dir.glob('*_desc-confounds_timeseries.tsv')) if func_dir.exists() else []
        status = 'DONE' if bold_t1w else 'not yet run'
        print(f'  sub-{sub} ses-{ses}: {status}')
        if bold_t1w:
            print(f'    BOLD (T1w): {bold_t1w[0].name}')
            print(f'    Confounds:  {confounds[0].name if confounds else "missing"}')

fMRIPrep outputs:
  sub-PAT16 ses-preop: DONE
    BOLD (T1w): sub-PAT16_ses-preop_task-rest_space-T1w_desc-preproc_bold.nii.gz
    Confounds:  sub-PAT16_ses-preop_task-rest_desc-confounds_timeseries.tsv
  sub-PAT16 ses-postop: DONE
    BOLD (T1w): sub-PAT16_ses-postop_task-rest_space-T1w_desc-preproc_bold.nii.gz
    Confounds:  sub-PAT16_ses-postop_task-rest_desc-confounds_timeseries.tsv


## 4. Step 3: QSIPrep DWI Preprocessing

QSIPrep preprocesses the diffusion MRI data:

1. **Head motion and eddy current correction** (FSL eddy)
2. **Susceptibility distortion correction** using the reverse phase-encoding
   (AP + PA) b0 pairs -- this is more accurate than SyN because actual
   field inhomogeneity information is available
3. **Bias field correction** (ANTs N4)
4. **Resampling** to 1.0 mm resolution in T1w-aligned (ACPC) space

The 1.0 mm output resolution matches the T1w grid, avoiding double
interpolation when mapping DTI metrics onto the FE mesh.

### Running QSIPrep

```bash
bash scripts/run_qsiprep.sh --subject PAT16 --session preop --nthreads 8
```

In [4]:
# Check QSIPrep outputs
qsiprep_dir = DERIV_ROOT / 'qsiprep'
print('QSIPrep outputs:')
print('=' * 60)
for sub in SUBJECTS:
    for ses in SESSIONS:
        dwi_dir = qsiprep_dir / f'sub-{sub}' / f'ses-{ses}' / 'dwi'
        dwi_files = list(dwi_dir.glob('*_desc-preproc_dwi.nii.gz')) if dwi_dir.exists() else []
        status = 'DONE' if dwi_files else 'not yet run'
        print(f'  sub-{sub} ses-{ses}: {status}')
        if dwi_files:
            print(f'    DWI: {dwi_files[0].name}')

QSIPrep outputs:
  sub-PAT16 ses-preop: DONE
    DWI: sub-PAT16_ses-preop_space-ACPC_desc-preproc_dwi.nii.gz
  sub-PAT16 ses-postop: DONE
    DWI: sub-PAT16_ses-postop_space-ACPC_desc-preproc_dwi.nii.gz


## 5. Step 4: MRtrix3 Reconstruction

MRtrix3 performs advanced diffusion modeling on the QSIPrep-preprocessed DWI:

### Constrained Spherical Deconvolution (CSD)

Unlike DTI (which assumes a single fiber per voxel), CSD resolves
**crossing fibers** by deconvolving the diffusion signal with a
single-fiber response function. The multi-shell multi-tissue (MSMT)
variant separates WM, GM, and CSF contributions.

For brain mechanics, the WM fiber orientation distributions (fODFs)
provide the local fiber architecture needed for **anisotropic constitutive
models** in the FE framework.

### DTI Scalar Maps

Even with CSD available, DTI scalar maps remain useful as proxies for
tissue mechanical properties:

| Metric | Mechanical Interpretation |
|--------|-------------------------|
| FA | Fiber alignment -> anisotropic stiffness |
| MD | Tissue density/hydration -> isotropic compliance |
| AD | Axial integrity -> longitudinal stiffness |
| RD | Myelin integrity -> transverse stiffness |

### Tractography + SIFT2

10 million streamlines are generated using the iFOD2 probabilistic
algorithm, then filtered with SIFT2 to produce quantitative
streamline weights that better reflect actual fiber density.

### Running MRtrix3 Reconstruction

```bash
bash scripts/run_mrtrix_recon.sh --subject PAT16 --session preop --nthreads 8
```

In [5]:
# Check MRtrix3 reconstruction outputs
mrtrix_dir = DERIV_ROOT / 'mrtrix_recon'
print('MRtrix3 reconstruction outputs:')
print('=' * 60)
for sub in SUBJECTS:
    for ses in SESSIONS:
        recon = mrtrix_dir / f'sub-{sub}' / f'ses-{ses}'
        fa = recon / 'fa.nii.gz'
        tck = recon / 'whole_brain.tck'
        wm_fod = recon / 'wm_fod_norm.mif'
        status = 'DONE' if fa.exists() else 'not yet run'
        print(f'  sub-{sub} ses-{ses}: {status}')
        if fa.exists():
            for f in [fa, tck, wm_fod]:
                size = f.stat().st_size / 1e6 if f.exists() else 0
                print(f'    {f.name}: {size:.1f} MB')

MRtrix3 reconstruction outputs:
  sub-PAT16 ses-preop: DONE
    fa.nii.gz: 5.7 MB
    whole_brain.tck: 7693.7 MB
    wm_fod_norm.mif: 976.3 MB
  sub-PAT16 ses-postop: DONE
    fa.nii.gz: 5.8 MB
    whole_brain.tck: 8018.2 MB
    wm_fod_norm.mif: 976.3 MB


## 6. Step 5: Co-registration to T1w Space

This step brings all derivatives into the fMRIPrep native T1w space:

1. **Rigid registration** (6-DOF, FLIRT): ACPC -> T1w
2. **DTI scalar maps**: trilinear interpolation
3. **fOD images**: spatial resampling + **spherical harmonic reorientation**
   (WM fODs need their SH coefficients rotated to match the new axes)
4. **Tractography**: each streamline vertex transformed by the 4x4 affine
5. **FreeSurfer volumes**: nearest-neighbor for labels, linear for brain.mgz

### Why SH reorientation matters

Unlike scalar maps (FA, MD), fiber orientation distributions are
directional quantities represented as spherical harmonics. A spatial
transform that rotates the coordinate axes must also rotate the
SH coefficients, otherwise fiber directions would point the wrong way.
MRtrix3's `mrtransform -reorient_fod yes` handles this automatically.

### Running Co-registration

```bash
bash scripts/register_to_t1w.sh --subject PAT16 --session preop --nthreads 8
```

### Inspecting with Freeview

```bash
bash scripts/freeview_all.sh --subject PAT16 --session preop
```

In [6]:
# Check registered T1w-space outputs
reg_dir = DERIV_ROOT / 'registered_t1w'
print('Registered T1w-space outputs:')
print('=' * 60)
expected = [
    'fa_space-T1w.nii.gz', 'md_space-T1w.nii.gz',
    'wm_fod_norm_space-T1w.mif', 'whole_brain_space-T1w.tck',
    'aparc+aseg_space-T1w.nii.gz'
]
for sub in SUBJECTS:
    for ses in SESSIONS:
        out = reg_dir / f'sub-{sub}' / f'ses-{ses}'
        found = sum(1 for f in expected if (out / f).exists())
        status = f'{found}/{len(expected)} files' if out.exists() else 'not yet run'
        print(f'  sub-{sub} ses-{ses}: {status}')
        if out.exists():
            for f in expected:
                fp = out / f
                mark = 'OK' if fp.exists() else 'MISSING'
                print(f'    [{mark}] {f}')

Registered T1w-space outputs:
  sub-PAT16 ses-preop: 5/5 files
    [OK] fa_space-T1w.nii.gz
    [OK] md_space-T1w.nii.gz
    [OK] wm_fod_norm_space-T1w.mif
    [OK] whole_brain_space-T1w.tck
    [OK] aparc+aseg_space-T1w.nii.gz
  sub-PAT16 ses-postop: 5/5 files
    [OK] fa_space-T1w.nii.gz
    [OK] md_space-T1w.nii.gz
    [OK] wm_fod_norm_space-T1w.mif
    [OK] whole_brain_space-T1w.tck
    [OK] aparc+aseg_space-T1w.nii.gz


## 6b. Step 5b: Longitudinal Registration (ANTs SyN)

Steps 1-5 operate **within** each session, producing T1w-space derivatives
for pre-op and post-op independently.  Step 5b bridges **between** sessions
using diffeomorphic (ANTs SyN) registration:

1. **Resection cavity masking** -- Automatic detection of the surgical void
   by comparing pre-op and post-op FreeSurfer segmentations (Tier 1, zero
   extra dependencies).  Optional Tier 2 uses the RESSEG deep-learning model
   (`pip install resseg`) for more precise masking.

2. **ANTs SyN registration** -- Diffeomorphic registration of pre-op T1w to
   post-op T1w, with the cavity excluded from the cost function (`-x` flag)
   so that SyN does not try to deform healthy tissue into the surgical void.

3. **Deformation field outputs**:
   - **Dense displacement field** u(x) -- encodes tissue motion at every voxel
   - **Jacobian determinant** J(x) -- local volume change (J > 1 = expansion)
   - **Displacement magnitude** |u(x)| in mm

4. **Warped pre-op derivatives** -- Pre-op T1w, DTI maps, and parcellation
   mapped into post-op space for proper voxel-wise comparison in Step 6.

For brain mechanics, the displacement field IS the empirical observable
that connects imaging to tissue mechanics:  it provides ground truth for
validating FE model predictions of brain deformation.

### Running Longitudinal Registration

```bash
# Default: SynthSeg-based cavity masking (zero extra deps)
bash scripts/register_longitudinal.sh --subject PAT16 --nthreads 8

# With RESSEG deep-learning cavity masking (requires: pip install resseg)
bash scripts/register_longitudinal.sh --subject PAT16 --resseg --nthreads 8

# Via the full pipeline
bash scripts/run_pipeline.sh --clinical-postop --parallel --threads 8
```

### Inspecting with Freeview

```bash
bash scripts/freeview_deformation.sh --subject PAT16
```

In [7]:
# Check longitudinal registration outputs
reg_dir = DERIV_ROOT / 'registered_t1w'
print('Longitudinal registration outputs (Step 5b):')
print('=' * 60)
long_expected = [
    'preop_to_postop_1Warp.nii.gz',
    'jacobian_preop_to_postop.nii.gz',
    'displacement_magnitude.nii.gz',
    'cavity_mask_postop.nii.gz',
    'T1w_preop_in_postop.nii.gz',
    'fa_preop_in_postop.nii.gz',
]
for sub in SUBJECTS:
    long_dir = reg_dir / f'sub-{sub}' / 'longitudinal'
    found = sum(1 for f in long_expected if (long_dir / f).exists())
    status = f'{found}/{len(long_expected)} files' if long_dir.exists() else 'not yet run'
    print(f'  sub-{sub}: {status}')
    if long_dir.exists():
        for f in long_expected:
            fp = long_dir / f
            mark = 'OK' if fp.exists() else 'MISSING'
            print(f'    [{mark}] {f}')

Longitudinal registration outputs (Step 5b):
  sub-PAT16: 6/6 files
    [OK] preop_to_postop_1Warp.nii.gz
    [OK] jacobian_preop_to_postop.nii.gz
    [OK] displacement_magnitude.nii.gz
    [OK] cavity_mask_postop.nii.gz
    [OK] T1w_preop_in_postop.nii.gz
    [OK] fa_preop_in_postop.nii.gz


## 7. Step 6: Pre-op vs Post-op Comparison

For each subject, we compare the registered derivatives between pre-operative
and post-operative sessions to quantify how tumor resection alters:

- **Tissue volumes**: ROI-level changes from aparc+aseg (ventricle enlargement, 
  cortical thinning near resection cavity)
- **Diffusion properties**: delta-FA and delta-MD maps revealing altered fiber 
  architecture and tissue hydration
- **Functional connectivity**: weakened or strengthened network connections

These changes directly impact brain mechanics modeling:

| Change | Mechanical Impact |
|--------|------------------|
| Decreased FA near cavity | Loss of anisotropic stiffness |
| Increased MD (edema) | Increased tissue compliance |
| Ventricle enlargement | Altered CSF boundary conditions |
| Tract disruption | Modified load paths in FE model |

### Running the Comparison

```bash
python scripts/compare_preop_postop.py --subject PAT16
```

In [8]:
import json

# Check for comparison results
print('Pre-op vs Post-op comparison results:')
print('=' * 60)
for sub in SUBJECTS:
    report_path = reg_dir / f'sub-{sub}' / 'preop_vs_postop' / 'comparison_report.json'
    if report_path.exists():
        with open(report_path) as f:
            report = json.load(f)
        analyses = report.get('analyses', {})
        print(f'\n  sub-{sub}: report found')
        if 'delta_fa' in analyses:
            fa = analyses['delta_fa']
            print(f'    delta-FA: mean={fa["mean"]:.4f}, std={fa["std"]:.4f}')
        if 'delta_md' in analyses:
            md = analyses['delta_md']
            print(f'    delta-MD: mean={md["mean"]:.6f}, std={md["std"]:.6f}')
        if 'mechanics_notes' in analyses:
            print('    Mechanics implications:')
            for note in analyses['mechanics_notes']:
                print(f'      - {note}')
    else:
        print(f'\n  sub-{sub}: not yet run')

Pre-op vs Post-op comparison results:

  sub-PAT16: report found
    delta-FA: mean=0.0001, std=0.2040
    delta-MD: mean=0.000057, std=0.001418
    Mechanics implications:
      - Mean MD increase of 0.000057 mm^2/s indicates tissue density/hydration changes, affecting isotropic compliance.
      - Left-Lateral-Ventricle volume changed by 9.8%, affecting CSF boundary conditions in FE model.
      - Right-Lateral-Ventricle volume changed by 38.2%, affecting CSF boundary conditions in FE model.
      - 95th percentile displacement of 6.2 mm indicates significant brain shift -- the FE reference configuration must account for this deformation.
      - 31564 voxels displaced > 10 mm -- large deformation formulation required in these regions.
      - 24.6% of voxels show expansion (J > 1.05), consistent with CSF space enlargement post-surgery.
      - Resection cavity volume: 18.6 cm3 -- creates a non-simply-connected FE domain requiring zero-stiffness / CSF boundary condition treatment.


## 8. Running the Full Pipeline

The master orchestration script processes all subjects and sessions:

```bash
# Full pipeline (all subjects, all sessions, parallel Steps 2+3)
bash scripts/run_pipeline.sh --parallel --threads 8 --mem-mb 16000

# Single subject/session
bash scripts/run_pipeline.sh --subjects PAT16 --sessions preop --threads 8

# Skip steps already completed
bash scripts/run_pipeline.sh --skip-freesurfer --skip-fmriprep --threads 8
```

### Freeview Inspection at Each Stage

```bash
# After FreeSurfer: inspect surfaces and parcellation
bash scripts/freeview_freesurfer.sh --subject PAT16 --session preop

# After co-registration: inspect all modalities in T1w space
bash scripts/freeview_all.sh --subject PAT16 --session preop

# After registration QC: overlay DWI ref on T1w
bash scripts/freeview_registration.sh --subject PAT16 --session preop

# Longitudinal deformation: displacement, Jacobian, cavity mask
bash scripts/freeview_deformation.sh --subject PAT16

# Longitudinal comparison: pre-op vs post-op side by side
bash scripts/freeview_preop_postop.sh --subject PAT16
```

## 9. Summary

The preprocessing pipeline transforms raw BIDS-formatted multimodal MRI
into a unified set of T1w-space derivatives suitable for brain mechanics
FE modeling. The key insight is that each modality contributes different
information to the digital twin:

- **T1w + FreeSurfer**: geometry and tissue type
- **DWI + CSD/DTI**: fiber architecture and material properties
- **fMRI**: functional network topology
- **Tumor masks**: pathological regions requiring special treatment
- **Longitudinal SyN (Step 5b)**: deformation fields, Jacobian,
  displacement -- empirical tissue motion for FE validation

The longitudinal (pre-op vs post-op) comparison reveals how surgery
changes the brain's mechanical state. The displacement field from
Step 5b provides ground-truth deformation data for validating
predictive FE models of brain shift and tissue remodeling.